# VisClick — Part C (step 1): pull repo + start data acquisition

**You already did:** GitHub repo, Google Drive folder `visclick`, Colab GPU.

**This notebook:** **pull** the repo, then **CLAY** + **UI-Vision**. **RICO:** from [interactionmining.org/rico](https://interactionmining.org/rico), place `rico_unique_uis.zip` in `visclick/data/raw/`, then run **C.2.1** below to unzip to `data/rico/`. Re-run only that cell after uploading. **Next:** `02` = Zenodo + VINS (02 skips RICO if already extracted). See `VisClick_Detailed_Plan.md` Part C.2.

**Drive path in Colab:** `My Drive/visclick` → `/content/drive/MyDrive/visclick`


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
# 1) Get latest code: clone if first time, else pull
import os, subprocess
REPO = "https://github.com/HiranMadhu/visclick.git"
ROOT = "/content/visclick"
if not os.path.isdir(os.path.join(ROOT, ".git")):
    subprocess.run(["git", "clone", REPO, ROOT], check=True)
    print("Cloned to", ROOT)
else:
    subprocess.run(["git", "-C", ROOT, "fetch", "origin"], check=False)
    subprocess.run(["git", "-C", ROOT, "pull", "--rebase", "origin", "main"], check=False)
    print("Pulled latest in", ROOT)


In [ ]:
# 2) Work inside the repo (configs, scripts, reports paths)
%cd /content/visclick


In [ ]:
# 3) Drive root (must match your Google Drive folder name)
import os
DRIVE = "/content/drive/MyDrive/visclick"
for sub in (
    "data/raw", "data/clay", "data/rico", "data/vins", "data/webui", "data/ui_vision",
    "data/unified", "data/source_train", "data/desktop_labeled", "data/desktop_test",
    "weights/baseline_source", "weights/experiments", "videos",
):
    os.makedirs(os.path.join(DRIVE, sub), exist_ok=True)
print("DRIVE =", DRIVE)


In [ ]:
# 4) Tools for downloads
!pip install -q "huggingface_hub[cli]>=0.20" tqdm


## C.2.2 — CLAY (labels; images need RICO later)

Clones into `Drive/.../data/raw/clay` and copies into `data/clay/`.


In [ ]:
import os, shutil, subprocess
DRIVE = "/content/drive/MyDrive/visclick"
raw = os.path.join(DRIVE, "data", "raw")
os.makedirs(raw, exist_ok=True)
os.chdir(raw)
if not os.path.isdir("clay"):
    subprocess.run(
        ["git", "clone", "https://github.com/google-research-datasets/clay.git", "clay"],
        check=True,
    )
clay_src = os.path.join(raw, "clay")
clay_dst = os.path.join(DRIVE, "data", "clay")
os.makedirs(clay_dst, exist_ok=True)
for name in os.listdir(clay_src):
    s, d = os.path.join(clay_src, name), os.path.join(clay_dst, name)
    if os.path.isdir(s):
        shutil.copytree(s, d, dirs_exist_ok=True)
    else:
        shutil.copy2(s, d)
print("CLAY ->", clay_dst)


## C.2.5 — UI-Vision (desktop benchmark, ~781 MB)

Saves under `data/ui_vision/`. Needs a Hugging Face **read** token if the dataset asks for it:  
Hub → Settings → Access Tokens → add token and run `huggingface-cli login` once in Colab.


In [ ]:
import os, subprocess, sys
DRIVE = "/content/drive/MyDrive/visclick"
out = os.path.join(DRIVE, "data", "ui_vision")
os.makedirs(out, exist_ok=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "huggingface_hub"], check=False)
# If 401: run in a new cell: !huggingface-cli login
r = subprocess.run(
    ["hf", "download", "ServiceNow/ui-vision", "--repo-type", "dataset", "--local-dir", out],
    check=False,
)
print("UI-Vision ->", out, "exit", r.returncode)


## C.2.1 — RICO (unzip only; you upload the zip first)

1. On your PC: download the **UI Screenshots and View Hierarchies** pack from [interactionmining.org/rico](https://interactionmining.org/rico) (or your mirror).  
2. In **Google Drive**, place the file as: `visclick/data/raw/rico_unique_uis.zip` (rename if needed).  
3. Run the next cell. It unzips into `visclick/data/rico/`. If the zip is missing, you will see `MANUAL_PENDING` — upload, then **re-run only the next cell** (no need to re-run CLAY/UI-Vision).

In [ ]:
import os, subprocess
DRIVE = "/content/drive/MyDrive/visclick"
raw = os.path.join(DRIVE, "data", "raw")
rico_zip = os.path.join(raw, "rico_unique_uis.zip")
rico_dir = os.path.join(DRIVE, "data", "rico")
os.makedirs(raw, exist_ok=True)
os.makedirs(rico_dir, exist_ok=True)
if os.path.isfile(rico_zip):
    r = subprocess.run(["unzip", "-o", "-q", rico_zip, "-d", rico_dir], capture_output=True)
    print("RICO unzip exit", r.returncode, "->", rico_dir)
    print("REPORT step = RICO | status = EXTRACTED | exit =", r.returncode, "| path =", rico_dir)
else:
    print("No", rico_zip, "— upload the file to that path, then re-run this cell.")
    print("REPORT step = RICO | status = MANUAL_PENDING | expected_zip =", rico_zip)

## Stop here for this session

**Check:** `visclick/data/clay`, `visclick/data/ui_vision`, and (after C.2.1) `visclick/data/rico` if you uploaded the zip.

**Copy for the report:** run the next code cell (`REPORT:` lines) and save the output — it maps to **`VisClick_Report_Data_Form.md` §0** (and parts of §1.2 / §1.3).

**Next Colab:** `notebooks/02_rico_zenodo_vins.ipynb` — Zenodo unified bundle + VINS (RICO unzip is optional there if you did C.2.1 here). Mount + `git pull` again is fine.

**Local:** Zotero + `reports/literature_table.csv` for papers (plan C.1).


In [ ]:
# REPORT (paste output into your notes / chat for VisClick_Report_Data_Form.md §0)
import os, platform, sys, subprocess

def _dir_stats(p):
    n, total = 0, 0
    if not os.path.isdir(p):
        return 0, 0
    for r, _d, files in os.walk(p):
        for f in files:
            fp = os.path.join(r, f)
            try:
                total += os.path.getsize(fp)
                n += 1
            except OSError:
                pass
    return n, total

DRIVE = "/content/drive/MyDrive/visclick"
paths = {
    "CLAY": os.path.join(DRIVE, "data", "clay"),
    "UI_VISION": os.path.join(DRIVE, "data", "ui_vision"),
    "RICO": os.path.join(DRIVE, "data", "rico"),
}
print("REPORT form_section = §0 (also §1.2 GPU / §1.3 python)")
print("REPORT notebook = 01_pull_and_data")
print("REPORT colab_python =", sys.version.split()[0])
for name, p in paths.items():
    nf, b = _dir_stats(p)
    print(f"REPORT dataset = {name} | path = {p} | files = {nf} | size_bytes = {b} | size_GB = {b / (1024**3):.3f}" if b else f"REPORT dataset = {name} | path = {p} | status = EMPTY_OR_MISSING | files = {nf}")
# GPU
try:
    r = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        capture_output=True,
        text=True,
        timeout=10,
    )
    if r.returncode == 0 and r.stdout.strip():
        print("REPORT §1.2 nvidia_smi =", r.stdout.strip().replace(chr(10), " | "))
    else:
        print("REPORT §1.2 nvidia_smi = NO_GPU_OR_FAILED exit", r.returncode)
except FileNotFoundError:
    print("REPORT §1.2 nvidia_smi = NOT_INSTALLED (CPU only?)")
print("REPORT end")
